In [ ]:
import re
import os 
import json
from pathlib import Path
import pdfplumber

In [ ]:
def extraer_filas_por_comision(path_pdf: str):
    """
    Extrae metadatos de cada comisión y agrupa las filas por comisión.
    Retorna un diccionario: {comision: [lista_de_filas]}.
    Cada fila contiene: numeroCamara, anioCamara, numeroSenado, anioSenado, tipoLey, titulo.
    """
    # Patrón para cabecera de fila
    header_pattern = re.compile(
        r'(?m)^(?P<index>\d+)\s+'
        r'(?P<numCam>\d{1,3})/(?P<anioCam>\d{4})C\s+'
        r'(?P<numSen>\d{1,3})/(?P<anioSen>\d{4})S\s+'
        r'(?P<tipoLey>Acto\s+Legislativo|Ley\s+Ordinaria|Ley\s+Estatutaria|Ley\s+Orgánica|ACU)',
        re.IGNORECASE
    )
    # Patrón para dividir bloques por cada nueva fila (encabezado)
    split_pattern = re.compile(
        r'(?m)(?=^\d+\s+\d{1,3}/\d{4}C\s+\d{1,3}/\d{4}S)' 
    )

    comisiones = {}
    current_comision = None
 
    with pdfplumber.open(path_pdf) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ''
            # Detectar comisión (persistente)
            for line in text.splitlines():
                if line.strip().upper().startswith('COMISION'):
                    parts = line.strip().split(None, 1)
                    current_comision = parts[1] if len(parts) > 1 else current_comision
                    break 
            # Filtrar encabezados y totales
            skip = ['SECRETARÍA GENERAL', 'REGISTRO Y TRÁMITE',
                    'DE LEY Y ACTOS LEGISLATIVOS', 'LEGISLATURA', 'TOTAL']
            filtered = []
            for line in text.splitlines():
                up = line.strip().upper()
                if any(up.startswith(pref) for pref in skip) or up.startswith('COMISION'):
                    continue
                filtered.append(line)
            content = '\n'.join(filtered)

            # Dividir por cada registro
            bloques = split_pattern.split(content)
            for bloq in bloques:
                bloq = bloq.strip()
                m_head = header_pattern.match(bloq)
                if not m_head:
                    continue
                datos = m_head.groupdict()
                resto = bloq[m_head.end():].strip()
                # Extraer título arrancando en POR MEDIO/POR EL/POR
                title_start = re.search(r'\b(POR\s+MEDIO|POR\s+EL|POR)\b', resto, re.IGNORECASE)
                raw = resto[title_start.start():] if title_start else resto
                # Recortar hasta cierre de comilla o al último punto
                if '”' in raw:
                    raw = raw.split('”', 1)[0]
                elif '.' in raw:
                    raw = raw.rsplit('.', 1)[0] + '.'
                # Limpieza y normalización
                title = raw.replace('“', '').replace('”', '')
                title = title.replace('<', '').replace('>', '')
                title = re.sub(r'\s+', ' ', title).strip(' "').rstrip('.')

                fila = {
                    'numeroCamara': datos['numCam'].lstrip('0'),
                    'anioCamara': datos['anioCam'],
                    'numeroSenado': datos['numSen'].lstrip('0'),
                    'anioSenado': datos['anioSen'],
                    'tipoLey': datos['tipoLey'].strip(),
                    'titulo': title
                }
                comisiones.setdefault(current_comision or 'SinComision', []).append(fila)
    return comisiones

In [60]:
def procesar_pdf_por_comision(path_pdf: str, carpeta_salida: str) -> None:
    """
    Procesa el PDF y guarda un archivo JSON por cada comisión con todas sus filas.
    """
    os.makedirs(carpeta_salida, exist_ok=True)
    comisiones = extraer_filas_por_comision(path_pdf)
    for comision, filas in comisiones.items():
        # Nombre seguro de archivo
        key = comision.lower().replace(' ', '_') if comision else 'sin_comision'
        out_path = Path(carpeta_salida) / f'{key}.json'
        with out_path.open('w', encoding='utf-8') as f:
            json.dump({ 'comision': comision, 'filas': filas }, f, indent=4, ensure_ascii=False)
    print(f"✅ Archivos JSON creados en: {Path(carpeta_salida).resolve()}")
    
if __name__ == "__main__":
    # Ejemplo de uso
    pdf_entrada = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\resource\2020_2021\2020 2021_LEGISLATURA_PROYECTOS_PRESENTADOS_COMISION_1.pdf"
    carpeta_json = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\2020_2021\ProyectosPresentadosComision"
    procesar_pdf_por_comision(pdf_entrada, carpeta_json)


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


✅ Archivos JSON creados en: C:\Users\juans\Documents\pro\Model-Extract-information\document\2020_2021\ProyectosPresentadosComision
